In [40]:
import os
import sys
import glob
import numpy as np
from tqdm import trange
from astropy.io import fits
from astropy.table import Table, vstack
from astropy.convolution import convolve, Gaussian1DKernel
import astropy.units as u
import astropy.coordinates as coord
import matplotlib
import matplotlib.pyplot as plt
from astropy.table import Column
from tqdm import trange
import pandas as pd
import matplotlib.ticker as mticker
import fitsio
from astropy.table import Table, vstack
from astropy import units as u
from astropy.coordinates import SkyCoord
from easyquery import Query, QueryMaker
from scipy.stats import binomtest
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LogNorm
from matplotlib.colors import ListedColormap, BoundaryNorm
import h5py
from astropy.cosmology import Planck18
import glob
from matplotlib.lines import Line2D

rootdir = '/global/u1/v/virajvm/'
sys.path.append(os.path.join(rootdir, 'DESI2_LOWZ/desi_dwarfs/code'))

from desi_lowz_funcs import make_subplots, match_c_to_catalog, print_radecs, process_img
from desi_lowz_funcs import calc_normalized_dist, sdss_rgb, get_scrollable_pdfs
from desi_lowz_funcs import find_objects_nearby, print_radecs
# from construct_dwarf_galaxy_catalogs import process_sga_matches

import warnings
from astropy.wcs import FITSFixedWarning

from desi_lowz_funcs import plot_2d_dist, make_subplots

%load_ext autoreload
%autoreload 2




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [41]:
cont_path = "/pscratch/sd/v/virajvm/desi_dwarf_catalogs/contaminants/contaminant_flags_dr1only.fits"


In [42]:
#!/usr/bin/env python
"""
Explore example objects from each contaminant class in the DESI DR1 dwarf
contaminant-flags catalog.

For every CONTAM_CLASS present in the file, this:
  1. prints the class distribution,
  2. pulls N example rows per class (strongest evidence first),
  3. decodes each object's CONTAM_BITMASK into human-readable evidence, and
  4. prints a Legacy Survey desi-spectrum viewer URL so you can load each one.

Reference: contaminant_flags README sections 4 and 6.
"""

import numpy as np
from astropy.table import Table

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
CONT_PATH = "/pscratch/sd/v/virajvm/desi_dwarf_catalogs/contaminants/contaminant_flags_dr1only.fits"

N_PER_CLASS = 3        # examples to show per class
SELECT = "strongest"   # "strongest" (conclusive first) or "random"
RANDOM_SEED = 0        # only used when SELECT == "random"

SPEC_URL = "https://www.legacysurvey.org/viewer/desi-spectrum/dr1/targetid{tid}"

# Confidence ordering: most decisive first.
CONF_RANK = {"conclusive": 0, "mh-provisional": 1, "suggestive": 2, "none": 3}

# Bit -> (short name, plain-English evidence). From README section 6, plus
# bit 20 (deep-spectrum tested) noted in section 3. Bits 16/17/20 only ever
# appear in the *_mh.fits file; harmless to keep here.
BIT_INFO = {
    0:  ("FLAGA_HIGH",          "two DR1 coadds disagree on z (dchi2>=100)"),
    1:  ("FLAGA_MED",           "two DR1 coadds disagree on z (dchi2>=40)"),
    2:  ("FLAGA_LOW",           "coadd z disagreement, lower conf (dchi2>=25)"),
    3:  ("RR_STAR_COADD",       "another coadd of this fiber fit as a STAR"),
    4:  ("GAIA_STAR",           "Gaia parallax/PM + quiescent spectrum"),
    5:  ("GAIA_NEIGHBOR_BLEND", "bright Gaia neighbor 1.2-4'' + FRACFLUX>0.35"),
    6:  ("MASKBITS_BRIGHT",     "Legacy bright-star/cluster mask touches source"),
    7:  ("GC_UCD_HOSTBOUND",    "pos/vel match to a massive 50MGC galaxy"),
    8:  ("UCD_NSC_FREE",        "compact + quiescent, no host association"),
    9:  ("SPEC_HIGHZ",          "spectroscopy: true system at a different z"),
    10: ("SPEC_STAR",           "spectroscopy: stellar absorption + cool cont."),
    11: ("SINGLE_LINE_Z",       "redshift rests on a single emission line"),
    12: ("PHOT_ARTIFACT",       "single-epoch imaging / bad fit (NOBS<=1, RCHISQ>4)"),
    13: ("DIST_NO_FLOW",        "cz<3000 km/s without flow-corrected distance"),
    14: ("NOT_PRIMARY_CLEAN",   "duplicate, not primary after fix"),
    15: ("SPEC_WRONGZ_SUSPECT", "wrong-z hinted by spectral break only"),
    16: ("MH_DISCREPANT_CONF",  "Matterhorn full-depth z differs, confidently"),
    17: ("MH_DISCREPANT_UNCERTAIN", "Matterhorn z differs, uncertain at full depth"),
    18: ("QSO_TARGET",          "DESI targeted it as a QSO candidate"),
    19: ("WISE_AGN_W12",        "AGN-like WISE color (W1-W2 >= 0.8)"),
    20: ("DEEP_SPEC_TESTED",    "full-depth spectrum adjudicated"),
}

# Known interesting objects from README section 8 (one-liner each).
WORKED_EXAMPLES = {
    39627793348170704: "K/M giant on a z=0.335 group galaxy (founding blend case)",
    39627715581578169: "second blend; fit directly as an M star at full depth",
    39633259813930361: "passive z~0.39 galaxy misfit as z=0.0025",
    39627649181550105: "globular cluster of NGC 4697 (dv=8 km/s, 12 kpc)",
    39628093597419171: "GC/UCD of M84, first misread as a foreground star",
    39633063008797204: "real z=0.018 galaxy, photometry killed by a G=13.3 star 4.8'' away",
    39627760284473918: "wrong-z 0.033->0.109, later confirmed to four digits",
}


# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------
def decode_bits(value):
    """Return list of (name, description) for every bit set in value."""
    value = int(value)
    return [BIT_INFO.get(b, (f"BIT{b}", "undocumented"))
            for b in range(value.bit_length()) if value & (1 << b)]


def as_str_col(col):
    """Normalize a FITS string column (may be bytes) to a python str array."""
    arr = np.asarray(col)
    if arr.dtype.kind == "S":
        arr = np.char.decode(arr, "utf-8")
    return arr.astype(str)


def scalar(row, name):
    """Fetch a column value from a Row; None if absent or masked/empty."""
    if name not in row.colnames:
        return None
    v = row[name]
    if v is np.ma.masked or (np.ma.isMaskedArray(v) and v.mask):
        return None
    if isinstance(v, (str, bytes, np.str_, np.bytes_)) and str(v).strip() == "":
        return None
    return v


def pick_examples(sub, n, how, rng):
    """Choose n row indices from sub-table `sub`."""
    if how == "random":
        return np.sort(rng.permutation(len(sub))[:n])
    # "strongest": conclusive confidence first, then highest suspicion score.
    conf = as_str_col(sub["CONFIDENCE"])
    rank = np.array([CONF_RANK.get(c, 9) for c in conf])
    susp = (np.asarray(sub["SUSPICION_SCORE"], float)
            if "SUSPICION_SCORE" in sub.colnames else np.zeros(len(sub)))
    order = np.lexsort((-susp, rank))   # primary: rank asc, secondary: susp desc
    return order[:n]


def fmt_num(v):
    """Pretty-print a possibly-missing/NaN numeric value."""
    if v is None:
        return "n/a"
    try:
        fv = float(v)
    except (TypeError, ValueError):
        return str(v)
    return "n/a" if not np.isfinite(fv) else f"{fv:g}"


  

In [5]:
t = Table.read(CONT_PATH)
print(f"Loaded {len(t):,} rows from:\n  {CONT_PATH}\n")
print("Columns:", ", ".join(t.colnames), "\n")

cls_col = as_str_col(t["CONTAM_CLASS"])
classes, counts = np.unique(cls_col, return_counts=True)

print("=" * 72)
print("CLASS DISTRIBUTION")
print("=" * 72)
for c, n in sorted(zip(classes, counts), key=lambda x: -x[1]):
    print(f"  {c:<24} {n:>9,}  ({100 * n / len(t):5.2f}%)")
print()

rng = np.random.default_rng(RANDOM_SEED)

for c in classes:
    sub = t[cls_col == c]
    idx = pick_examples(sub, N_PER_CLASS, SELECT, rng)
    print("=" * 72)
    print(f"CLASS: {c}   (n={len(sub):,}, showing {len(idx)})")
    print("=" * 72)
    for i in idx:
        row = sub[int(i)]
        tid = int(row["TARGETID"])

        print(f"\n  TARGETID {tid}")
        print(f"    spectrum        : {SPEC_URL.format(tid=tid)}")
        print(f"    Z (catalog)     : {fmt_num(scalar(row, 'Z'))}")

        other_z = scalar(row, "FLAGA_OTHER_Z")
        if other_z is not None and np.isfinite(float(other_z)) and float(other_z) > 0:
            print(f"    FLAGA_OTHER_Z   : {fmt_num(other_z)}   (disagreeing coadd z)")
        tier = scalar(row, "FLAGA_TIER")
        if tier is not None:
            tier_s = (as_str_col(np.array([tier]))[0]
                      if np.asarray(tier).dtype.kind in "SU" else fmt_num(tier))
            print(f"    FLAGA_TIER      : {tier_s}")

        conf = as_str_col(np.array([row["CONFIDENCE"]]))[0]
        print(f"    confidence      : {conf}")
        print(f"    suspicion_score : {fmt_num(scalar(row, 'SUSPICION_SCORE'))}")
        print(f"    anomaly_pct     : {fmt_num(scalar(row, 'ANOMALY_PCT'))}")
        print(f"    knn_contam_pct  : {fmt_num(scalar(row, 'KNN_CONTAM_PCT'))}")
        print(f"    gaia_match      : {fmt_num(scalar(row, 'GAIA_MATCH'))}")
        for util in ("Z_RELIABLE", "DIST_RELIABLE", "DWARF_PRIMARY_CLEAN"):
            if util in sub.colnames:
                print(f"    {util:<15} : {int(row[util])}")

        bm = int(row["CONTAM_BITMASK"])
        print(f"    bitmask ({bm}) :")
        bits = decode_bits(bm)
        if bits:
            for name, desc in bits:
                print(f"        - {name}: {desc}")
        else:
            print("        (no bits set)")

        if tid in WORKED_EXAMPLES:
            print(f"    >> worked example: {WORKED_EXAMPLES[tid]}")
    print()



Loaded 469,715 rows from:
  /pscratch/sd/v/virajvm/desi_dwarf_catalogs/contaminants/contaminant_flags_dr1only.fits

Columns: TARGETID, Z, CONTAM_BITMASK, CONTAM_CLASS, CONFIDENCE, SUSPICION_SCORE, ANOMALY_PCT, KNN_CONTAM_PCT, FLAGA_TIER, FLAGA_OTHER_Z, Z_RELIABLE, DIST_RELIABLE, DWARF_PRIMARY_CLEAN, GAIA_MATCH 

CLASS DISTRIBUTION
  unvetted                   444,874  (94.71%)
  clean                       19,978  ( 4.25%)
  photometry-corrupted         4,270  ( 0.91%)
  gc-ucd-nsc-candidate           492  ( 0.10%)
  blend                           62  ( 0.01%)
  catastrophic-z                  20  ( 0.00%)
  gc-ucd-nsc                      10  ( 0.00%)
  star                             9  ( 0.00%)

CLASS: blend   (n=62, showing 3)

  TARGETID 39627715581576842
    spectrum        : https://www.legacysurvey.org/viewer/desi-spectrum/dr1/targetid39627715581576842
    Z (catalog)     : 0.00222588
    FLAGA_OTHER_Z   : 0.35151   (disagreeing coadd z)
    FLAGA_TIER      : 1
    confidence

In [ ]:
## ok these flags are pretty good!! I like them, but they are flagging some real objects, need to look into this more but very promising

## 

In [45]:
# Path to the catalog
filename = "/pscratch/sd/v/virajvm/desi_dwarf_catalogs/dr1/v1.0/desi_dr1_dwarf_catalog.fits"

# Option 1: load the MAIN extension directly as an Astropy Table
samp = Table.read(filename, hdu="MAIN")

samp_fspec = Table.read("/pscratch/sd/v/virajvm/catalog_dr1_dwarfs/desi_dr1_dwarfs.fits")


In [44]:
print(len(samp))

469715


In [46]:
len(samp_fspec)

461488

In [ ]:
## I see there is a slighlt difference between the two!

In [32]:
missing_tgids = np.loadtxt("/pscratch/sd/v/virajvm/desi_dwarf_catalogs/dr1/v1.0/missing_fastspec_targetids.txt",dtype=int)

In [33]:
missing_tgids[:5]

array([39627328841582075, 39627339843247169, 39627345564271632,
       39627351125918747, 39627374890846555])

In [34]:
len(missing_tgids)

8227

In [35]:
mask = np.isin(samp["TARGETID"].data, missing_tgids)
missing = samp[mask]

In [36]:
import fitsio, numpy as np
from astropy.table import Table


In [38]:
redux =  "/global/cfs/cdirs/desi/spectro/redux"

In [39]:
for row in missing[:50]:
    s, p, h = row['SURVEY'], row['PROGRAM'], row['HEALPIX']
    rr = f"{redux}/iron/healpix/{s}/{p}/{h//100}/{h}/redrock-{s}-{p}-{h}.fits"
    rid = fitsio.read(rr, 'REDSHIFTS', columns='TARGETID')
    print(row['TARGETID'], s, p, h, 'PRESENT' if row['TARGETID'] in rid else 'ABSENT')

##so these are not in the 

39627328841582075 main bright 36263 PRESENT
39627339843247169 main bright 17408 PRESENT
39627345564271632 main bright 17410 PRESENT
39627351125918747 main bright 16834 PRESENT
39627374890846555 main bright 36297 PRESENT
39627380771266209 main bright 36214 PRESENT
39627385896704008 main bright 17417 PRESENT
39627392397873996 main bright 22538 PRESENT
39627397573642096 main bright 36539 PRESENT
39627409594517156 main bright 36285 PRESENT
39627415030334208 main bright 36542 PRESENT
39627415504291893 main bright 36301 PRESENT
39627415621734223 main bright 22562 PRESENT
39627421288242211 main bright 36321 PRESENT
39627427177038707 main bright 36312 PRESENT
39627438694597362 main bright 36633 PRESENT
39627444151393062 main bright 17456 PRESENT
39627444537262965 main bright 36634 PRESENT
39627444788921279 main bright 36309 PRESENT
39627444889588465 main bright 22554 PRESENT
39627450006637130 main bright 17445 PRESENT
39627450392515719 main bright 36634 PRESENT
39627450631591426 main bright 36